# SFT — fine-tune SynapseGPT (v2)

Runs `sft.py` on the v2 dataset (12 sources, 175,245 train / 3,569 val ChatML
records). Loads the pretrained checkpoint, applies the `SFT_DATA_MIX` with
masked (response-only) loss, and writes checkpoints to
`$SYNAPSE_DIR/sft_checkpoints/` on Drive. The model to ship is
**`sft_best.pth`** — saved automatically whenever overall val improves.

**Needs an A100 runtime** (Runtime → Change runtime type → A100 GPU). All real
logic lives in `sft.py` and the shared `synapse_model.py`; this notebook just
wires up Drive + the repo and launches it.


In [ ]:
# Mount Drive (Colab) and set SYNAPSE_DIR
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    SYNAPSE_DIR = '/content/drive/MyDrive/synapse'
except ImportError:
    import os
    SYNAPSE_DIR = os.environ.get('SYNAPSE_DIR') or os.path.abspath('./synapse')
import os
os.environ['SYNAPSE_DIR'] = SYNAPSE_DIR
print('SYNAPSE_DIR =', SYNAPSE_DIR)

In [ ]:
# Clone or update the repo (brings sft.py AND the shared synapse_model.py)
import os, subprocess, sys
REPO_DIR = '/content/synapse_repo'
REPO_URL = 'https://github.com/ajencinas/synapse.git'
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print('repo exists — pulling latest')
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    print(f'cloning {REPO_URL} -> {REPO_DIR}')
    subprocess.run(['git', 'clone', '--depth=1', REPO_URL, REPO_DIR], check=True)
print('REPO_DIR =', REPO_DIR)
assert os.path.isfile(os.path.join(REPO_DIR, 'sft', 'sft.py')), 'sft.py missing — check branch'
assert os.path.isfile(os.path.join(REPO_DIR, 'synapse_model.py')), 'synapse_model.py missing — check branch'

In [ ]:
# GPU check
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU — switch Runtime → Change runtime type → GPU before training.')

In [ ]:
# Preflight: confirm the inputs sft.py needs are present on Drive
import os
need = {
    'tokenizer':        os.path.join(SYNAPSE_DIR, 'tokenizer_out', 'tokenizer.json'),
    'pretrain manifest':os.path.join(SYNAPSE_DIR, 'manifests', 'training_latest.json'),
    'sft registry':     os.path.join(SYNAPSE_DIR, 'manifests', 'sft_data_registry.json'),
    'pretrain ckpt':    os.path.join(SYNAPSE_DIR, 'checkpoints', 'synapse_2b_d2560_l28.pth'),
}
ok = True
for label, path in need.items():
    present = os.path.exists(path)
    ok = ok and present
    print(('  OK ' if present else 'MISS ') + f'{label}: {path}')
assert ok, 'missing prerequisites above — run the data pipeline / upload the pretrain checkpoint first'
print('\nAll prerequisites present.')

In [ ]:
# Train. Checkpoints land in $SYNAPSE_DIR/sft_checkpoints/ (on Drive — persists).
# Toggle SFT_COMPILE=0 to fall back to eager if torch.compile misbehaves.
!cd {REPO_DIR} && SYNAPSE_DIR={SYNAPSE_DIR} SFT_COMPILE=1 python sft/sft.py

In [ ]:
# Results: baseline (pre-SFT) vs final per-source val loss
import json, os
m = json.load(open(os.path.join(SYNAPSE_DIR, 'manifests', 'sft_training_latest.json')))
base = (m.get('results') or {}).get('baseline_eval') or {}
last = (m.get('results') or {}).get('latest_eval') or {}
print(f"epoch={m['epoch']}  step={m['curr_step']}/{m['total_steps']}\n")
print(f"{'source':12s} {'baseline':>9} {'final':>9} {'delta':>9}")
for k in sorted(set(base) | set(last)):
    b, l = base.get(k), last.get(k)
    d = (l - b) if (b is not None and l is not None) else None
    print(f"{k:12s} {('%.3f'%b) if b is not None else '   -':>9} "
          f"{('%.3f'%l) if l is not None else '   -':>9} "
          f"{('%+.3f'%d) if d is not None else '   -':>9}")

## Notes

- **Checkpoints**: the full checkpoint (`sft_latest.pth`, with optimizer) is written
  to local `/content/sft_work/` for speed, then mirrored to
  `$SYNAPSE_DIR/sft_checkpoints/` on Drive in the background (every
  `SFT_DRIVE_EVERY` steps + each epoch). Per-epoch model-only snapshots
  (`sft_epochN.pth`) and **`sft_best.pth`** (best overall val — the model to
  ship) are pushed durably (the push is waited on, never skipped).
- **Resume**: just re-run — it auto-resumes (`stage=="sft"` + tokenizer match),
  preferring the local full checkpoint, then the Drive copy, then a model-only
  snapshot, then a fresh start from the pretrain checkpoint. (v1 checkpoints
  were moved to `sft_checkpoints/v1_7source/` so v2 starts clean.)
- **What to expect at baseline**: tool_use / tool_negative val around 3–6
  (tokens `<|tool_call|>`/`<|tool_result|>` are unseen at pretrain) — they must
  drop sharply. The 7 v1 sources should end at/below their v1 finals
  (tulu3 1.745 · metamath 0.281 · dolly 2.003 · alpaca 1.429 · samsum 1.431 ·
  opencode 0.260 · oasst1 1.532).
- **RunPod / bare VM**: don't use this notebook — `git clone` and run
  `bash sft/run_sft_on_vm.sh` (pulls inputs via rclone, trains, mirrors
  checkpoints back to Drive).
